# Algorytm Scale-Invariant Feature Transform (SIFT)
_Kurs "Analiza danych obrazowych i multimedialnych" 26L_

- **Zespół 4:** Cong Minh Vu, Mateusz Szulc, Szymon Ochnio, Mateusz Kwiatkowski
- **Data realizacji:** maj 2026

### Opis algorytmu

Algorytm **SIFT (Scale-Invariant Feature Transform)** służy do detekcji i szczegółowego opisu lokalnych cech obrazu, co pozwala na rozpoznawanie i dopasowywanie tych samych obiektów na różnych zdjęciach.

W wyniku swojego działania wyznacza on punkty charakterystyczne obrazu (key points), które cechują się:
- niewrażliwością na jednorodne zmiany skali obrazu i jego rotację,
- częściową niewrażliwością na zmiany oświetlenia i położenie kamery (perspektywę).

Proces wykrywania punktów charakterystycznych składa się z kilku etapów:
1. **Detekcja potencjalnych punktów charakterystycznych** - obraz jest wielokrotnie rozmywany i skalowany. Badanie różnic między rozmyciami pozwala na wykrycie potencjalnych lokalnych cech w różnych skalach. Zapewnia to niewrażliwość na jednorodne zmiany skali obrazu.
2. **Lokalizacja punktów charakterystycznych** - wyznacza się dokładną lokalizację punktów w dziedzinie subpixeli, a następnie odrzuca punkty o niskiej stabilności oraz przydatności w zadaniu dopasowania obiektów - takie znajdujące się na krawędziach oraz o niskim kontraście.
3. **Przypisanie orientacji** - przypisuje punktom dominujący kierunek gradientu ich otoczenia. Zapewnia to niewrażliwość na rotację obrazu.
4. **Budowa deskryptorów** - opisuje punkty poprzez charakterystyczne wektory 128 cech utworzone poprzez lokalne zastosowanie metody HOG w ich otoczeniu. Zapewnia to wysoką rozróżnialność punktów i upraszcza proces poszukiwania odpowiadających punktów między obrazami.

Algorytm SIFT posiada liczne zastosowania i jest powszechnie wykorzystywany m.in. do: rozpoznawania obiektów, śledzenia ruchu, wirtualnej rzeczywistości (AR), tworzenia panoram (stitching), modelowania 3D.

**Przydatne źródła:**
- [Oficjalny tutorial OpenCV dla metody SIFT](https://docs.opencv.org/4.x/da/df5/tutorial_py_sift_intro.html)
- [Artykuł oryginalny: *David G. Lowe (2004) "Distinctive Image Features from Scale-Invariant Keypoints"*](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf)
- [Opis algorytmu SIFT (Wikipedia)](https://en.wikipedia.org/wiki/Scale-invariant_feature_transform)
- [Repozytorium kodu OpenCV z przykładami](https://github.com/opencv/opencv)

### Przedstawienie algorytmu SIFT - krok po kroku (uproszczone)

Poniżej zostało przedstawione działanie algorytmu SIFT krok po kroku z odpowiednimi wizualizacjami. Pewne kroki algorytmu zostały uproszczone w celu lepszego przedstawienia kluczowych elementów algorytmu

#### Krok 0. Importy i funkcje pomocnicze

In [ ]:
%pip install opencv-python matplotlib numpy scipy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def show_img(img: np.ndarray, title="Obraz", cmap='gray', figsize=(10, 8)):
    plt.figure(figsize=figsize)

    if img.ndim == 3:  # Konwersja BGR do RGB dla Matplotlib
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap)

    plt.title(title, fontsize=16)
    plt.axis('off')
    plt.show()

#### Krok 1. Wczytywanie obrazu

Wczytujemy obraz, na których będziemy pracować. Przekształcamy go w skalę szarości, ponieważ SIFT operuje na zmianach jasności, wobec czego kolor jest niepotrzebny. Dla łatwiejszych obliczeń wartości jasności pikseli zostają znormalizowane do zakresu <0;1>.

In [ ]:
# USTAW NA ŚCIEŻKĘ WŁASNEGO OBRAZU
IMAGE_PATH = 'data/ref.png'

image = cv2.imread(IMAGE_PATH)

if image is None:
    raise "Błąd: Nie znaleziono obrazu!"

image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0

fig, ax = plt.subplots(1, 2, figsize=(15, 12))
ax[0].imshow(image[:, :, ::-1])
ax[0].set_title("Obraz - Oryginał (kolor)")
ax[0].axis('off')
ax[1].imshow(image_gray, cmap='gray')
ax[1].set_title("Obraz - Skala szarości")
ax[1].axis('off')

plt.tight_layout()
plt.show()

#### Krok 2. Przekształcenie w przestrzeni skali (piramida Gaussjanów)

Obrazy często zawierają obiekty w znacząco różnych rozmiarach, wobec czego przetwarzanie ich w jednolity sposób mogłoby być nieefektywne - w tym celu stosuje się przetwarzanie w wielu skalach. Algorythm SIFT dokonuje tego poprzez kaskadowe rozmywanie obrazu filtrem Gaussa, co symuluje oddalanie się od obiektu.

Jako że rosnące rozmywanie obrazu powoduje utratę szczegółów, przetwarzanie obrazu w tym samym rozmiarze powoduje wyłącznie zbędne koszty obliczeniowe. W tym celu tworzy się tzw. **oktawy** - zestawy obrazów o jednakowym rozmiarze, ale stale rosnącym poziomie rozmycia. Co pewną liczbę obrazów, tworzy się nową oktawę z rozmazanym obrazem pomniejszonym dwukrotnie, a następnie kontynuuje się cały proces.

Współczynnik rozmycia ($\sigma$) rośnie geometrycznie z każdym krokiem o stałą wartość $k$. Możemy zauważyć, jak wraz z rosnącym rozmyciem znikają drobne detale, a zostają tylko główne, "silne" cechy kształtu.

_Ważne: w tej prezentacji zostanie wyznaczona wyłącznie jedna oktawa_

In [ ]:
# USTAW WŁASNE WARTOŚCI - domyślne zgodne z artykułem Lowe
# liczba interwałów - określa, ile znaczących poziomów skali będzie w oktawie
NO_OF_SCALES = 3
# bazowy współczynnik rozmycia
SIGMA_BASE = 1.6

images_per_octave = NO_OF_SCALES + 3
k = 2 ** (1.0 / NO_OF_SCALES)

gaussian_pyramid = []
titles_gauss = []

for i in range(images_per_octave):
    current_sigma = k ** i * SIGMA_BASE
    blurred = cv2.GaussianBlur(image_gray, (0, 0), sigmaX=current_sigma, sigmaY=current_sigma)

    gaussian_pyramid.append(blurred)
    titles_gauss.append(f"Skala {i}\n$\\sigma$={current_sigma:.2f}")

# wizualizacja
n_cols = 3
n_rows = (images_per_octave + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 5 * n_rows), dpi=120)
for ax, img_layer, title in zip(axes.ravel(), gaussian_pyramid, titles_gauss):
    ax.imshow(img_layer, cmap='gray')
    ax.set_title(title, fontsize=12)
    ax.axis('off')
for ax in axes.ravel()[images_per_octave:]:
    ax.axis('off')
plt.suptitle("Piramida Gaussjanów (oktawa 1.)", fontsize=16)
plt.tight_layout()
plt.show()

#### Krok 3. Wyznaczenie różnicy Gaussjanów (Difference of Gaussians - DoG)

W celu efektywnego wykrycia punktów charakterystycznych odejmujemy od siebie sąsiadujące ze sobą obrazy z utworzonej przed chwilą piramidy Gaussjanów. Dla 6 obrazów w oktawie otrzymamy 5 obrazów różnicowych.

Operacja ta działa jak zaawansowany filtr krawędziowy (przybliżenie Laplasjanu Gaussa). Usuwa ona obszary o jednolitej jasności, a uwypukla miejsca, w których następuje gwałtowna zmiana tekstury (rogi, krawędzie, plamy).

In [ ]:
dog_pyramid = []
titles_dog = []

for i in range(images_per_octave - 1):
    dog = cv2.subtract(gaussian_pyramid[i + 1], gaussian_pyramid[i])
    dog_pyramid.append(dog)
    titles_dog.append(f"DoG {i}\n(Skala {i + 1} - Skala {i})")

# wizualizacja
n_cols = 3
n_rows = (len(dog_pyramid) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 5 * n_rows), dpi=120)
for ax, dog_layer, title in zip(axes.ravel(), dog_pyramid, titles_dog):
    ax.imshow(dog_layer, cmap='gray')
    ax.set_title(title, fontsize=12)
    ax.axis('off')
for ax in axes.ravel()[len(dog_pyramid):]:
    ax.axis('off')
plt.suptitle("Różnica Gaussjanów (DoG)", fontsize=16)
plt.tight_layout()
plt.show()

#### Krok 4. Detekcja potencjalnych punktów charakterystycznych (ekstrema przestrzeni skali)

Aby punkt mógł zostać uznany za lokalną cechę (niewrażliwą na zmianę skali), musi być lokalnym ekstremum nie tylko na swoim obrazie, ale również w wymiarze skali.

Pojedynczy piksel na wybranym poziomie (np. `DoG 1`) jest porównywany z **26 sąsiadami**:
* **8** najbliższych sąsiadów na tym samym poziomie (`DoG 1`),
* **9** sąsiadów na poziomie poniżej (`DoG 0`),
* **9** sąsiadów na poziomie powyżej (`DoG 2`).

Poniżej została wykonana detekcja ekstremów **wyłącznie** w obrazie różnicowym `DoG 1`, odrzucając przy tym punkty o zbyt niskim kontraście, które mogłyby zostać uznane za szum. Wyniki nanosimy na oryginalny kolorowy obraz.

In [ ]:
# USTAW WŁASNE WARTOŚCI
# próg kontrastu
CONTRAST_THRESHOLD = 0.03

from scipy.ndimage import maximum_filter, minimum_filter

dog_stack = np.stack(dog_pyramid[0:3], axis=-1)

is_max = maximum_filter(dog_stack, size=(3, 3, 3)) == dog_stack
is_min = minimum_filter(dog_stack, size=(3, 3, 3)) == dog_stack
is_extrema = is_max | is_min

current_image_idx = 1

extrema_mask = is_extrema[:, :, current_image_idx]
stability_mask = np.abs(dog_pyramid[current_image_idx]) > CONTRAST_THRESHOLD

final_mask = extrema_mask & stability_mask

key_points = list(zip(*np.where(final_mask)))

print(f"Liczba wykrytych punktów charakterystycznych na poziomie DoG 1: {len(key_points)}")

# wizualizacja
output_img = image.copy()

for y, x in key_points:
    cv2.circle(output_img, (x, y), radius=3, color=(0, 0, 255), thickness=-1)

show_img(output_img, title="Wykryte ekstrema w DoG 1", figsize=(10, 8))

#### Krok 5. Eliminacja punktów na krawędziach

Z poprzedniego kroku uzyskaliśmy listę potencjalnych punktów charakterystycznych, ale nadal nie wszystkie z nich są stabilne. Algorytm SIFT generuje wiele punktów na prostych krawędziach, które nie są dobrymi lokalnymi cechami. Możemy wyobrazić sobie kropkę na linii prostej - przy nieznacznym przesunięciu wzdłuż tej linii, otoczenie będzie wyglądać identycznie, co znacząco utrudnia dopasowanie punktów między obrazami.

SIFT rozwiązuje ten problem, analizując macierz Hessego w każdym wykrytym punkcie na poziomie DoG. Macierz ta opisuje krzywiznę obrazu w dwóch prostopadłych kierunkach. Jeżeli stosunek większej krzywizny do mniejszej jest zbyt duży, oznacza to, że punkt mamy do czynienia z krawędzią, a nie z narożnikiem lub "plamką". Taki punkt jest wówczas odrzucany.

In [ ]:
# USTAW WŁASNE WARTOŚCI
# próg eliminacji krawędzi
R = 10

threshold_edge = ((R + 1.0) ** 2) / R
dog_image = dog_pyramid[1]

# obliczamy pochodne cząstkowe rzędu drugiego (Hessian) za pomocą filtru Sobela
dxx = cv2.Sobel(dog_image, cv2.CV_64F, 2, 0, ksize=3)
dyy = cv2.Sobel(dog_image, cv2.CV_64F, 0, 2, ksize=3)
dxy = cv2.Sobel(dog_image, cv2.CV_64F, 1, 1, ksize=3)

filtered = []
rejected = []

H, W = dog_image.shape

for y, x in key_points:
    # pomijamy punkty zbyt blisko krawędzi obrazu
    if y == 0 in (0, H - 1) or x in (0, W - 1):
        continue

    Dxx, Dyy, Dxy = dxx[y, x], dyy[y, x], dxy[y, x]
    tr = Dxx + Dyy
    det = (Dxx * Dyy) - (Dxy ** 2)

    # odrzucamy punkty z ujemnym wyznacznikiem (krzywizny mają różne znaki)
    if det <= 0:
        rejected.append((y, x))
        continue

    score = (tr ** 2) / det

    if score < threshold_edge:
        filtered.append((y, x))
    else:
        rejected.append((y, x))

print(f"Przed filtracją: {len(key_points)} punktów")
print(f"Po filtracji: {len(filtered)} punktów")
print(f"Odrzucono: {len(rejected)} punktów ({(len(rejected) / len(key_points)) * 100:.1f}%)")

# wizualizacja
output_image = image.copy()
for y, x in filtered:
    cv2.circle(output_image, (x, y), radius=3, color=(0, 255, 0), thickness=-1)

for y, x in rejected:
    cv2.circle(output_image, (x, y), radius=3, color=(0, 0, 255), thickness=-1)

show_img(output_image, title="Stabilne (zielone) i odrzucone (czerwone) punkty charakterystyczne", figsize=(10, 8))

#### Krok 6: Przypisanie orientacji

Z poprzednich kroków uzyskaliśmy zbiór punktów charakterystycznych odpornych na zmianę skali. Aby zapewnić odporność na obrót obrazu, algorytm analizuje obszar wokół każdego punktu charakterystycznego na odpowiednio rozmytym obrazie z piramidy Gaussa w celu wykrycia "dominującego" kierunku gradientu w obszarze. Odkryty kierunek staje się główną orientacją punktu.

Takie działanie umożliwi odpowiednią reprezentację punktu niezależnie od obrotu obrazu. Poniżej przedstawiono przypisanie orientacji dla **wyłącznie jednego** ze znalezionych punktów kluczowych obrazu:

In [ ]:
# dla punktu z DoG(1) = L(x,y,1) - L(x,y,0) badamy otoczenie w L(x,y,0)
gaussian_image = gaussian_pyramid[0]

# wyznaczamy siłę / kierunek gradientu dla pikseli
gx = cv2.Sobel(gaussian_image, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(gaussian_image, cv2.CV_64F, 0, 1, ksize=3)

magnitude, angle = cv2.cartToPolar(gx, gy, angleInDegrees=True)

# badamy dla pierwszego (losowego) punktu
pt_y, pt_x = filtered[0]
radius = 8

H, W = gaussian_image.shape

assert pt_y - radius > 0 and pt_y + radius < H and \
       pt_x - radius > 0 and pt_x + radius < W

mag_window = magnitude[pt_y - radius: pt_y + radius, pt_x - radius: pt_x + radius]
ang_window = angle[pt_y - radius: pt_y + radius, pt_x - radius: pt_x + radius]

hist, bins = np.histogram(ang_window.flatten(), bins=36, range=(0, 360), weights=mag_window.flatten())

dominant_angle_bin = np.argmax(hist)
dominant_angle = bins[dominant_angle_bin]

print(f"Analizujemy punkt (X:{pt_x}, Y:{pt_y})")
print(f"Dominujący kąt gradientu: {dominant_angle:.1f}°")

# WIZUALIZACJA

# szersze otoczenie (wycinek ok. 200x200 pikseli wokół punktu)
ctx_radius = 100

y_start, y_end = max(0, pt_y - ctx_radius), min(H, pt_y + ctx_radius)
x_start, x_end = max(0, pt_x - ctx_radius), min(W, pt_x + ctx_radius)

plt.figure(figsize=(5, 5))
plt.imshow(gaussian_image[y_start:y_end, x_start:x_end], cmap='gray')
plt.plot(pt_x - x_start, pt_y - y_start, 'ro', markersize=5)
plt.title("Szersze otoczenie punktu (~200x200)")
plt.axis('off')
plt.show()

# zoom na okno 16x16 z wektorem orientacji
plt.figure(figsize=(4, 4))
plt.imshow(gaussian_image[pt_y - radius: pt_y + radius, pt_x - radius: pt_x + radius], cmap='gray')
plt.plot(radius, radius, 'ro', markersize=5)

arrow_len = 5
dx = arrow_len * np.cos(np.deg2rad(dominant_angle))
dy = arrow_len * np.sin(np.deg2rad(dominant_angle))
plt.arrow(radius, radius, dx, dy, color='red', head_width=0.8, head_length=1.2)

plt.title("Zoom na okno 16x16 z orientacją")
plt.axis('off')
plt.show()

# wizualizacja
plt.figure(figsize=(10, 4))
plt.bar(bins[:-1], hist, width=10, align='edge', color='orange', edgecolor='black')
plt.axvline(dominant_angle, color='red', linestyle='dashed', linewidth=2,
            label=f'Główna orientacja ({dominant_angle:.1f}°)')
plt.title("Histogram kierunków gradientu dla pojedynczego punktu")
plt.xlabel("Kąt (stopnie)")
plt.ylabel("Siła (suma modułów gradientu)")
plt.legend()
plt.show()

#### Krok 7: Podsumowanie (Wizualizacja z OpenCV)

Ostatni krok algorytmu SIFT wyznacza dla znalezionych punktów charakterystycznych deskryptory, które zapewniają wysoką unikalność i umożliwiają efektywne dopasowywanie punktów między obrazami. W celu wyznaczenia deskryptora wokół punktu tworzy się siatkę 4x4. W każdym z 16 podregionów wyznaczany jest osobny histogram kierunków gradientu (8 koszyków). Daje to 16 * 8 = **128-wymiarowy wektor cech**.

Z uwagi na to, że etap ten ma znaczenie wyłącznie dla porównywania obrazów, zostanie on pominięty w tej prezentacji. Poniżej przedstawimy pełne wyniki działania algorytmu, wykorzystując wbudowaną implementację SIFT z biblioteki `OpenCV`.

**Ważne:** poniższa konfiguracja OpenCV jest zgodna z parametrami z poprzednich kroków w celu zapewnienia porównywalnych wyników:
* współczynnik rozmycia `sigma = 1.6` (z kroku 2)
* próg kontrastu `contrast_threshold = 0.03` (z kroku 4)
* próg eliminacji krawędzi `R = 10` (z kroku 5)

*Uwaga:* Powyższy kod analizował ***WYŁĄCZNIE*** jedną oktawę oraz jedną różnicę Gaussjanów. OpenCV automatycznie przeskaluje obraz wielokrotnie, tworząc pełną wielopoziomową piramidę, dlatego znajdzie znacznie więcej punktów charakterystycznych.

In [ ]:
cv_image = (image_gray * 255).astype(np.uint8)
sift_custom = cv2.SIFT_create(
    nfeatures=1000,  # liczba znalezionych (najlepszych cech) - nfeatures=0 oznacza brak limitu
    nOctaveLayers=3,  # liczba interwałów w oktawie (wymusza 6 obrazów Gaussa)
    contrastThreshold=0.03,  # próg kontrastu
    edgeThreshold=10,  # próg eliminacji krawędzi
    sigma=1.6  # bazowy współczynnik rozmycia
)

keypoints_cv, descriptors_cv = sift_custom.detectAndCompute(cv_image, None)

final_img = cv2.drawKeypoints(
    image,
    keypoints_cv,
    None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    color=(0, 0, 255)
)

show_img(final_img, title=f"Wynik końcowy (OpenCV): {len(keypoints_cv)} najlepszych punktów",
         figsize=(12, 10))

### Opis implementacji

Nasza implementacja tworzy otoczkę wokół struktur udostępnianych przez bibliotekę `OpenCV` w celu zapewnienia prostej konfiguracji i jednolitego interfejsu. Udostępnione zostają dwie główne klasy i dwie pomocnicze:

- Klasa `SiftFeatureExtractor` odpowiada za wczytywanie obrazów oraz ekstrakcję punktów kluczowych i deskryptorów przy użyciu algorytmu SIFT. Dodatkowo zawiera wbudowaną metodę do łatwej wizualizacji wykrytych cech bezpośrednio na zdjęciu.
- Klasa `SiftFeatureMatcher` zajmuje się łączeniem odpowiadających sobie deskryptorów z dwóch obrazów i odrzucaniem błędnych par za pomocą testu proporcji. Odpowiada również za obliczanie macierzy homografii, co pozwala na precyzyjne narysowanie ramki wokół rozpoznanego obiektu na scenie.
- Klasa pomocnicza `SiftFeatureExtractorConfig` przechowuje hiperparametry dla algorytmu SIFT. Pozwala w przejrzysty sposób dostroić takie wartości jak maksymalna liczba wykrywanych cech, progi kontrastu czy parametry rozmycia Gaussa.
- Klasa pomocnicza `SiftFeatureMatcherConfig` definiuje ustawienia procesu dopasowywania cech pomiędzy różnymi obrazami. Przechowuje wybór algorytmu wyszukującego (np. szybki FLANN lub dokładny Brute-Force) oraz kluczowe progi tolerancji dla testu Lowe'a i algorytmu RANSAC.

Zostały one przygotowane poniżej:

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Tuple, List, Optional
from enum import Enum


@dataclass
class SiftFeatureExtractorConfig:
    feature_count: int = None  # maksymalna liczba cech do znalezienia (None = brak limitu)
    no_of_scales: int = 3  # liczba poziomów skali w oktawie
    contrast_threshold: float = 0.03  # Próg kontrastu (odrzucanie słabych cech)
    edge_threshold: float = 10.0  # Próg krawędzi (odrzucanie cech na krawędziach)
    sigma_base: float = 1.6  # Rozmycie Gaussa dla pierwszej oktawy


class SiftFeatureExtractor:
    def __init__(self, config: SiftFeatureExtractorConfig):
        self.config = config
        self.sift = cv2.SIFT_create(
            nfeatures=config.feature_count if config.feature_count is not None else 0,
            nOctaveLayers=config.no_of_scales,
            contrastThreshold=config.contrast_threshold,
            edgeThreshold=config.edge_threshold,
            sigma=config.sigma_base
        )

    def load_image(self, path: str, as_gray: bool = False) -> np.ndarray:
        """Ładuje obraz z dysku."""
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Nie można załadować obrazu z: {path}")

        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if as_gray else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def extract_features(self, image: np.ndarray) -> Tuple[Tuple[cv2.KeyPoint, ...], np.ndarray]:
        """Znajduje punkty kluczowe i oblicza ich deskryptory."""
        gray_image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if image.ndim == 3 else image
        keypoints, descriptors = self.sift.detectAndCompute(gray_image, None)

        return keypoints, descriptors

    def visualize_keypoints(self, image: np.ndarray, keypoints: Tuple[cv2.KeyPoint, ...], title: str = "Keypoints",
                            rich_display: bool = False):
        """Rysuje punkty kluczowe na obrazie."""
        img_with_kp = cv2.drawKeypoints(
            image, keypoints, None,
            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS if rich_display else cv2.DRAW_MATCHES_FLAGS_DEFAULT,
            color=(0, 255, 0)
        )

        plt.figure(figsize=(10, 8))
        plt.imshow(img_with_kp)
        plt.title(f"{title} (liczba cech: {len(keypoints)})")
        plt.axis('off')
        plt.show()

In [ ]:
class FeatureMatcherType(Enum):
    FLANN = 1,
    BFMatcher = 2


@dataclass
class SiftFeatureMatcherConfig:
    matcher_type: FeatureMatcherType = FeatureMatcherType.FLANN  # typ algorytmu dopasowania
    lowe_ratio: float = 0.75  # próg dla testu Lowe'a (zwykle 0.7 - 0.8)
    ransac_threshold: float = 5.0  # próg błędu reprojekcji dla RANSAC (przy homografii)


class SiftFeatureMatcher:
    def __init__(self, config: SiftFeatureMatcherConfig):
        self.config = config

        if config.matcher_type == FeatureMatcherType.FLANN:
            # Parametry dla FLANN i SIFT (który używa wektorów zmiennoprzecinkowych L2)
            FLANN_INDEX_KDTREE = 1
            index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
            search_params = dict(checks=50)
            self.matcher = cv2.FlannBasedMatcher(index_params, search_params)
        elif config.matcher_type == FeatureMatcherType.BFMatcher:
            # SIFT używa normy L2 (odległość euklidesowa)
            self.matcher = cv2.BFMatcher(cv2.NORM_L2)
        else:
            raise ValueError("Nieobsługiwany algorytm dopasowania")

    def match_features(self, desc1: np.ndarray, desc2: np.ndarray) -> List[cv2.DMatch]:
        """Dopasowuje deskryptory wykorzystując test knn (k=2) i test proporcji Lowe'a."""
        if desc1 is None or desc2 is None:
            return []

        raw_matches = self.matcher.knnMatch(desc1, desc2, k=2)
        good_matches = []

        for match_pair in raw_matches:
            if len(match_pair) == 2:
                m, n = match_pair
                if m.distance < self.config.lowe_ratio * n.distance:
                    good_matches.append(m)

        return good_matches

    def find_homography(self, kp1: Tuple[cv2.KeyPoint, ...], kp2: Tuple[cv2.KeyPoint, ...],
                        matches: List[cv2.DMatch]) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        """Oblicza macierz homografii wykorzystując RANSAC."""
        if len(matches) < 4:
            print("Zbyt mało dopasowań, aby obliczyć homografię (wymagane minimum 4).")
            return None, None

        # Wyciągnij współrzędne dopasowanych punktów
        src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

        # Znajdź homografię z użyciem RANSAC
        H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, self.config.ransac_threshold)
        return H, mask

    def visualize_matches(self, img1: np.ndarray, kp1: Tuple[cv2.KeyPoint, ...],
                          img2: np.ndarray, kp2: Tuple[cv2.KeyPoint, ...],
                          matches: List[cv2.DMatch], mask: Optional[np.ndarray] = None,
                          draw_homography_box: bool = False, H: Optional[np.ndarray] = None):
        """Wizualizuje dopasowania oraz (opcjonalnie) ramkę rzutowaną przez homografię."""

        matches_mask = mask.ravel().tolist() if mask is not None else None

        draw_params = dict(
            matchColor=(0, 255, 0),
            singlePointColor=(255, 0, 0),
            matchesMask=matches_mask,
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
        )

        if draw_homography_box and H is not None:
            h, w = img1.shape[:2]

            pts = np.float32([[0, 0], [0, h - 1], [w - 1, h - 1], [w - 1, 0]]).reshape(-1, 1, 2)
            dst = cv2.perspectiveTransform(pts, H)
            img2_boxed = img2.copy()
            img2_boxed = cv2.polylines(img2_boxed, [np.int32(dst)], True, (255, 0, 0), 3, cv2.LINE_AA)
            img_matches = cv2.drawMatches(img1, kp1, img2_boxed, kp2, matches, None, **draw_params)
        else:
            img_matches = cv2.drawMatches(img1, kp1, img2, kp2, matches, None, **draw_params)

        plt.figure(figsize=(15, 10))
        plt.imshow(img_matches)
        plt.title(f"Dopasowania (Znaleziono: {len(matches)}" +
                  (f", Inliers: {sum(matches_mask)}" if matches_mask else "") + ")")
        plt.axis('off')
        plt.show()

### Prezentacja wyników działania
Zaczniemy od przedstawienia detekcji lokalnych cech obrazu. Po wczytaniu obrazu wyznaczamy jego punkty charakterystyczne i zwizualizujemy je na tym samym obrazie:

In [ ]:
extractor_config = SiftFeatureExtractorConfig()
sift = SiftFeatureExtractor(extractor_config)

In [ ]:
IMAGE_PATH = 'data/ref.png'  # replace with valid file path to your image
image1 = sift.load_image(IMAGE_PATH)

features1, descriptors1 = sift.extract_features(image1)

sift.visualize_keypoints(image1, features1, title='SIFT - detekcja punktów charakterystycznych', rich_display=True)

Punkty charakterystyczne zostały zaznaczone na obrazie za pomocą czerwonych obręczy przedstawiających również wielkość oraz orientację punktów, co jest możliwe dzięki zastosowaniu parametru `rich_display`.

---

Następnie przedstawimy przykład dopasowania punktów między dwoma obrazami. W tym celu dokonamy ekstrakcji cech w drugim obrazie i wyszukamy odpowiednie dopasowania między obrazami:

In [ ]:
IMAGE_PATH = 'data/mod.png'  # replace with valid file path to your image
image2 = sift.load_image(IMAGE_PATH)

features2, descriptors2 = sift.extract_features(image2)

sift.visualize_keypoints(image2, features2, title='SIFT - obraz drugi z cechami', rich_display=True)

In [ ]:
matcher_config = SiftFeatureMatcherConfig()
matcher = SiftFeatureMatcher(matcher_config)

In [ ]:
matches = matcher.match_features(descriptors1, descriptors2)

matcher.visualize_matches(image1, features1, image2, features2, matches)

Wykorzystując dopasowane punkty charakterystyczne, możemy obliczyć macierz homografii, która pozwala zlokalizować obiekt na scenie niezależnie od zmiany jego perspektywy, skali czy rotacji. Algorytm RANSAC automatycznie odrzuca przy tym błędne dopasowania, opierając obliczenia wyłącznie na tych poprawnych geometrycznie. Dzięki temu możemy przekształcić narożniki obrazu referencyjnego i narysować na zdjęciu wynikowym precyzyjną ramkę, wskazującą dokładne położenie szukanego wzorca.

In [ ]:
H, mask = matcher.find_homography(features1, features2, matches)
matcher.visualize_matches(image1, features1, image2, features2, matches, mask=mask, H=H, draw_homography_box=True)

### Weryfikacja eksperymentalna

TODO

### Wnioski

TODO